# 14 — Cross-cohort stability (manuscript Figure 3)

For **one state set** (chosen below; HMM, mean-shift or threshold), the two claims of the
cross-cohort section and the figure behind them. `tools/stability_sweep.py` runs the same
summaries over every state set, to choose it.

1. **The transition structure replicates across cohorts and conditions** (panel C): the
   correlation between cohorts' mean p(j | i) divided by the noise ceiling (two disjoint halves
   of a cohort's people). The cohorts differ in age, scanner, TR and run length, so their LEVELS
   are not expected to match; their structure is. With `BALANCE = "time"` every cohort
   contributes the same hours of scan in every draw — people and stimulus duration balanced
   together, no cohort left out.
2. **Some properties depend on context, others do not** (panel D): movie vs rest in the SAME
   people (Cam-CAN, HCP), the one contrast that isolates context, as |d_av| in tiers: < 0.2
   stable, 0.2–0.5 moderate, > 0.5 context-dependent (Cohen's small / medium).

Panels A.1 (the reference to remember), A.2 (movie vs rest) and B (every cohort's levels) are
descriptive. **HMM caveat:** projected cohorts are labelled with the Viterbi path, which uses the
training transition matrix as a prior; a set without a temporal model (mean-shift, threshold)
has no such prior.

In [ ]:
import sys
from pathlib import Path

# THE PACKAGE, which is not pip-installed inside the analysis image. A Jupyter
# kernel starts with sys.path[0] = the notebook's own directory, so without
# this `import fmri_decomposition` fails from notebooks/.
#
# The REPO ROOT and nothing else. `notebooks/` used to be on this list too,
# back when the table and figure modules lived beside the .ipynb files; that
# part is gone and must not come back -- a directory on sys.path can shadow a
# real package, which is how an ImportError turns into the wrong module.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct

# ---- THE STATE SET. Any clusterer stage 4b wrote works: HMM (HMM1_/HMM2_), mean-shift
# (MeanShift_...), threshold (ThresholdCluster_...); the list printed below shows what exists.
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_pca3_8"          # e.g. "MeanShift_umap3_9", "ThresholdCluster_umap3_8"
SAVE_DIR = None                    # e.g. "figures/14" to save PNG + SVG

# ---- DATA BALANCE for the replication (panel C). "time": every cohort contributes the same
# hours of scan, drawn at random as whole scans (runs, films, episodes) and pooled as transition
# counts -- people and stimulus duration balanced together. None: each cohort's mean as it is.
BALANCE  = "time"
HOURS    = 10                      # per cohort and draw; CNeuroMod (~48 h) and Cam-CAN movie (~80 h) allow it
N_BOOT   = 200                     # draws (95% CI of panel C) and bootstrap of panel D
from fmri_decomposition.figures import cohorts as cp

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",   # None = entropy_rate_bits as stored (plug-in)
    # label -> the cohort= directories it reads, and its role in the state fit. A new cohort is one line.
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",     # a task= that IS this (whole name, any case) is rest: Rest, REST1..4 (HCP); every other is movie
    # figure 1: movie vs rest
    alpha=0.10,                    # BH-FDR
    fc_task=1.0,                   # |log2FC| for the graph (1 = twofold)
    fc_dyn=0.25,                   # |log2FC| for the dynamics
    edge_top=12,                   # the mean's strongest arrows, per task
    # figures 2 and 4: equivalence
    sesoi_d=0.2,                   # TOST bounds, in pooled SDs
    top_transitions=8,             # figure 2: the strongest transitions shown
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; e.g.")
for _, r in sets.head(8).iterrows():
    print(f"  space=dict(atlas={r.atlas!r}, window_s={r.window_s}, states={r.states!r})")
co = ct.Cohorts(cfg)
# What is read, from where, and as which kind: check this before reading any figure.
co.tasks()

## Figure 3

A.1 reference graph (every cohort and condition, each once) · A.2 movie vs rest in the same
people (blue = differs, paired, FDR) · B every cohort's movie: occupancy, p(j | i), dynamic
metrics (mean, 90% CI; filled = training cohort) · C replication of the structure over the noise
ceiling (95% CI over draws) · D context tiers (bootstrap 95% CI over people).

In [ ]:
graphs = co.graphs()
marks, _ = co.task_change()
agree = co.agreement(co.dyn)
st = co.stability(n_boot=N_BOOT)
pairs, ceil = co.replication("movie", balance=BALANCE, hours=HOURS, n_boot=N_BOOT)
cp.fig_figure3(co, graphs, marks, st, pairs, agree)

In [ ]:
display(ceil.round(3))                       # each cohort's noise ceiling (split-half r)
display(pairs.round(3))                      # panel C
st[["variable", "task_d", "task_lo", "task_hi", "tier", "n_paired"]].round(3)   # panel D

## Supplementary — individual consistency

Do people keep their rank between rest and movie? ICC(3,1) on cohort-demeaned values (y) against
the context shift (x). It depends on the data per person (HCP: four runs of each; Cam-CAN: one
short run), so read it per cohort (`icc_<cohort>`), and as a limit on per-person metrics rather
than a property of the brain.

In [ ]:
PLANE_VARIABLES = ["transition_fit", *co.dyn, *co.occ]
cp.fig_stability_plane(co, st, PLANE_VARIABLES)

In [ ]:
st.filter(regex="^variable$|^icc").round(2)

## Supplementary — variation between films (three pages)

Does a property depend on **which** film is watched? One row per movie task (film, episode, run),
grouped by cohort, each with its mean and 95% CI over its viewers. Three pages: the dynamic
metrics, the occupancy of every state, and the transitions.

Each film is compared with **the other films of its own cohort**:

* **within person** (filled marker) when the cohort's films share viewers (HCP runs, CNeuroMod
  episodes): each person's value in this film against their own mean over their other films,
  paired — the clean test of a film effect;
* **between people** (hollow) when each film had its own viewers (NNDb): Welch against the
  cohort's other films pooled — film and person are confounded, read it as descriptive;
* a cohort with one film (Cam-CAN) has nothing to compare.

A **red ring** (or a red outline in the matrices) marks a film that **differs** from the others
(BH-FDR < `alpha` over every film and variable). `equivalent` in the table is the TOST at ±
`sesoi_d` SD; with few viewers per film it is rarely reached, so the figures mark differences.
Count-based variables (distinct transitions) depend on run length, which differs between films.

In [ ]:
fs = co.film_summary()
fs[fs["variable"].isin(["transition_fit", *co.dyn])].round(3)

In [ ]:
DYN_VARIABLES = ["transition_fit", *co.dyn]
cp.fig_films_forest(co, fs, DYN_VARIABLES, co.rest_means(DYN_VARIABLES),
                    title="Supplementary Fig. Sa — dynamic metrics, film by film", name="films_dynamics")

In [ ]:
cp.fig_films_forest(co, fs, co.occ, co.rest_means(co.occ),
                    title="Supplementary Fig. Sb — occupancy of every state, film by film", name="films_occupancy",
                    labels={o: f"S{o[4:]} (share of time)" for o in co.occ})

In [ ]:
cp.fig_films_matrices(co, fs)

In [ ]:
# How many films differ from the others, per variable (within-person films only)
w = fs[fs["within"]]
w.groupby("variable")["differs"].mean().sort_values(ascending=False).round(2).head(20)

### The three comparisons on one scale (for the text)

|d| and Cohen's tiers (< 0.2 stable, 0.2–0.5 moderate, > 0.5 context-dependent) for: rest vs movie
in the same people; film vs film in the same people (HCP runs, CNeuroMod episodes: the film is the
only thing that changes); movie across cohorts (different people, films, scanners and ages change
together: it can show that a property is alike despite all that, not why it differs).


In [ ]:
cs = co.context_summary(st, fs)
cs.round(2)


## Check — rest and movie per cohort

Every cohort's own diagrams (the input of A.1, A.2 and B), with the movie-vs-rest changes per
cohort. A check, not a figure of the paper.

In [ ]:
marks_all, dyn = co.task_change()
cp.fig_diagrams(co, graphs, marks_all, dyn)